# 16_E8 - Al-Kafri official pairing and mask decode rescue

Objetivo: intentar rescatar el dataset axial Al-Kafri/Sudirman usando metadata y estructura oficial, antes de descartarlo para segmentación axial.

Este notebook **no entrena modelos**. Solo:
- busca documentos, CSVs y scripts oficiales extraídos;
- reconstruye índices de imágenes y GT;
- perfila valores reales de máscaras PNG;
- intenta pairing por metadata oficial (`Slices Numbers.csv`, `T1_Subfolders.csv`, `T2_Subfolders.csv`, etc.);
- genera overlays por valor/clase para revisión manual;
- produce una decisión técnica trazable.


In [ ]:

import importlib.util, subprocess, sys
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("pydicom")
ensure_package("skimage", "scikit-image")

import json, re, zipfile, math, os
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.transform import resize
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive no montado automáticamente:", repr(exc))

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


In [ ]:

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
ALKAFRI_ROOT = PFI_ROOT / "data" / "AXIAL_ALKAFRI"
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
NESTED_ROOT = EXTRACTED_ROOT / "_nested"

E7_ROOT = PFI_ROOT / "results" / "E7_alkafri_axial_curated_subset"
E8_ROOT = PFI_ROOT / "results" / "E8_alkafri_official_pairing"
FIGURES_ROOT = PFI_ROOT / "figures"
DOCS_ROOT = PFI_ROOT / "docs"

for p in [E8_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("NESTED_ROOT:", NESTED_ROOT)
print("E8_ROOT:", E8_ROOT)

if not NESTED_ROOT.exists():
    raise FileNotFoundError(f"No existe {NESTED_ROOT}. Primero ejecutar extracción/inventario Al-Kafri.")


In [ ]:

def norm_case(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    m = re.search(r"(\d{1,4})", s)
    if not m:
        return None
    n = int(m.group(1))
    if 1 <= n <= 9999:
        return f"{n:04d}"
    return None

def infer_modality(text):
    s = str(text).upper()
    if re.search(r"(^|[^A-Z0-9])T1([^A-Z0-9]|$)", s) or "T1_" in s or "_T1" in s:
        return "T1"
    if re.search(r"(^|[^A-Z0-9])T2([^A-Z0-9]|$)", s) or "T2_" in s or "_T2" in s:
        return "T2"
    return None

def infer_disc(text):
    s = str(text).upper()
    m = re.search(r"D\s*([345])", s)
    return int(m.group(1)) if m else None

def safe_read_csv(path):
    for enc in ["utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, engine="python", encoding_errors="ignore")

def rel(p):
    try:
        return str(Path(p).relative_to(ALKAFRI_ROOT))
    except Exception:
        return str(p)

def read_docx_text(path):
    try:
        with zipfile.ZipFile(path) as zf:
            xml = zf.read("word/document.xml").decode("utf-8", errors="ignore")
        text = re.sub(r"<[^>]+>", " ", xml)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    except Exception as exc:
        return f"[No se pudo leer docx: {exc!r}]"


In [ ]:

# 1) Inventario de documentos, metadata y scripts oficiales
all_files = [p for p in NESTED_ROOT.rglob("*") if p.is_file()]
docs = [p for p in all_files if p.suffix.lower() in [".docx", ".txt", ".md"]]
csvs = [p for p in all_files if p.suffix.lower() == ".csv"]
matlab = [p for p in all_files if p.suffix.lower() in [".m", ".mat"]]

inventory_df = pd.DataFrame([{
    "file_path": str(p),
    "relative_path": rel(p),
    "name": p.name,
    "suffix": p.suffix.lower(),
    "size_bytes": p.stat().st_size,
} for p in docs + csvs + matlab]).sort_values(["suffix", "relative_path"])

inventory_df.to_csv(E8_ROOT / "E8_official_files_inventory.csv", index=False)
print("docs:", len(docs), "csvs:", len(csvs), "matlab/mat:", len(matlab))
display(inventory_df.head(80))

doc_rows = []
for p in docs:
    txt = read_docx_text(p) if p.suffix.lower() == ".docx" else p.read_text(errors="ignore")
    doc_rows.append({"file_path": str(p), "relative_path": rel(p), "chars": len(txt), "preview": txt[:1500]})
docs_preview_df = pd.DataFrame(doc_rows)
docs_preview_df.to_csv(E8_ROOT / "E8_docs_preview.csv", index=False)
display(docs_preview_df.head(20))

m_rows = []
for p in [x for x in matlab if x.suffix.lower() == ".m"][:80]:
    txt = p.read_text(errors="ignore")
    hits = bool(re.search(r"Slice|slice|D3|D4|D5|Ground|Label|SegNet|T1|T2|png|ima", txt))
    m_rows.append({"file_path": str(p), "relative_path": rel(p), "chars": len(txt), "keyword_hits": hits, "preview": txt[:1200]})
matlab_preview_df = pd.DataFrame(m_rows)
matlab_preview_df.to_csv(E8_ROOT / "E8_matlab_scripts_preview.csv", index=False)
display(matlab_preview_df.head(20))


In [ ]:

# 2) Cargar metadata CSV oficial
metadata_tables = {}
for p in csvs:
    try:
        df = safe_read_csv(p)
        metadata_tables[p.name] = df
        print("\n---", p.name, df.shape, "---")
        print("path:", rel(p))
        display(df.head(10))
    except Exception as exc:
        print("ERROR leyendo", p, repr(exc))

metadata_summary_df = pd.DataFrame([{
    "name": name,
    "rows": len(df),
    "columns": list(df.columns),
} for name, df in metadata_tables.items()])
metadata_summary_df.to_csv(E8_ROOT / "E8_metadata_csv_summary.csv", index=False)
display(metadata_summary_df)


In [ ]:

# 3) Cargar índices de imágenes axiales y GT generados en E7
image_index_path = E7_ROOT / "E7_alkafri_axial_image_case_index.csv"
gt_index_path = E7_ROOT / "E7_alkafri_gt_case_index.csv"

if image_index_path.exists() and gt_index_path.exists():
    image_df = pd.read_csv(image_index_path, low_memory=False)
    gt_df = pd.read_csv(gt_index_path, low_memory=False)
    print("Usando índices E7 existentes.")
else:
    raise FileNotFoundError("Faltan índices E7. Ejecutar notebook 15/patch de indexación antes de este notebook.")

image_df["case_id_norm"] = image_df["case_id"].apply(norm_case)
gt_df["case_id_norm"] = gt_df["case_id"].apply(norm_case)
image_df["modality"] = image_df["modality"].fillna(image_df["image_relative_path"].apply(infer_modality))
gt_df["modality"] = gt_df["modality"].fillna(gt_df["gt_relative_path"].apply(infer_modality))
gt_df["disc_or_slice_id"] = gt_df["disc_or_slice_id"].fillna(gt_df["gt_relative_path"].apply(infer_disc))

print("image_df:", image_df.shape)
print("gt_df:", gt_df.shape)
display(image_df.head())
display(gt_df.head())

image_df.to_csv(E8_ROOT / "E8_image_index_normalized.csv", index=False)
gt_df.to_csv(E8_ROOT / "E8_gt_index_normalized.csv", index=False)


In [ ]:

# 4) Parsear Slices Numbers.csv a formato largo, de forma robusta
def parse_slices_metadata(metadata_tables):
    rows = []
    for table_name, df in metadata_tables.items():
        if "slice" not in table_name.lower() and "slices" not in table_name.lower():
            continue
        for ridx, row in df.iterrows():
            row_text = " ".join([str(v) for v in row.values if pd.notna(v)])
            case_id = None
            modality = infer_modality(row_text)
            disc = infer_disc(row_text)
            for col in df.columns:
                if re.search(r"case|patient|subject|id", str(col), re.I):
                    case_id = norm_case(row[col])
                    if case_id:
                        break
            if not case_id:
                case_id = norm_case(row_text)

            # Columnas D3/D4/D5 explícitas
            has_disc_cols = False
            for col, v in row.items():
                d_col = infer_disc(col)
                if d_col and pd.notna(v):
                    has_disc_cols = True
                    for m in re.finditer(r"\b\d{1,2}\b", str(v)):
                        n = int(m.group(0))
                        if 1 <= n <= 60:
                            rows.append({
                                "source_table": table_name,
                                "source_row": ridx,
                                "case_id": case_id,
                                "modality": modality,
                                "disc_id": d_col,
                                "slice_number": n,
                                "source_column": str(col),
                                "source_value": str(v),
                            })
            # Fallback si no hay columnas D explícitas
            if not has_disc_cols:
                for col, v in row.items():
                    if pd.isna(v):
                        continue
                    d_val = infer_disc(v) or disc
                    for m in re.finditer(r"\b\d{1,2}\b", str(v)):
                        n = int(m.group(0))
                        if 1 <= n <= 60:
                            rows.append({
                                "source_table": table_name,
                                "source_row": ridx,
                                "case_id": case_id,
                                "modality": modality,
                                "disc_id": d_val,
                                "slice_number": n,
                                "source_column": str(col),
                                "source_value": str(v),
                            })
    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame()

slices_long_df = parse_slices_metadata(metadata_tables)
slices_long_df.to_csv(E8_ROOT / "E8_slices_numbers_long_candidate.csv", index=False)
print("slices_long_df:", slices_long_df.shape)
display(slices_long_df.head(50))
if len(slices_long_df):
    display(slices_long_df.groupby(["source_table"], dropna=False).size().reset_index(name="n"))
    display(slices_long_df.groupby(["modality", "disc_id"], dropna=False).size().reset_index(name="n"))


In [ ]:

# 5) Perfil de valores reales de máscaras PNG
def read_mask_raw(path):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return arr

def value_profile(mask_path, max_values=30):
    arr = read_mask_raw(mask_path)
    if arr.ndim == 3:
        flat = arr.reshape(-1, arr.shape[-1])
        vals, counts = np.unique(flat, axis=0, return_counts=True)
        values = [tuple(map(int, v.tolist())) for v in vals]
    else:
        vals, counts = np.unique(arr.reshape(-1), return_counts=True)
        values = [int(v) for v in vals]
    total = int(np.prod(arr.shape[:2]))
    prof = []
    order = np.argsort(counts)[::-1]
    for idx in order[:max_values]:
        val = values[idx]
        cnt = int(counts[idx])
        prof.append({"value": str(val), "count": cnt, "ratio": cnt / total})
    return arr.shape, prof

gt_png = gt_df[
    gt_df["extension"].astype(str).str.lower().eq(".png")
    & gt_df["case_id_norm"].notna()
    & gt_df["modality"].notna()
].copy()

sample_gt = (
    gt_png.sort_values(["gt_type", "modality", "case_id_norm", "disc_or_slice_id"])
    .groupby(["gt_type", "modality"], group_keys=False)
    .head(25)
    .copy()
)

profile_rows = []
for _, row in tqdm(sample_gt.iterrows(), total=len(sample_gt), desc="mask value profiles"):
    try:
        shape, prof = value_profile(row["gt_file_path"])
        for item in prof:
            profile_rows.append({
                "gt_file_path": row["gt_file_path"],
                "gt_relative_path": row["gt_relative_path"],
                "case_id": row["case_id_norm"],
                "modality": row["modality"],
                "gt_type": row["gt_type"],
                "disc_or_slice_id": row["disc_or_slice_id"],
                "raw_shape": str(shape),
                **item
            })
    except Exception as exc:
        profile_rows.append({"gt_file_path": row["gt_file_path"], "error": repr(exc)})

mask_value_profile_df = pd.DataFrame(profile_rows)
mask_value_profile_df.to_csv(E8_ROOT / "E8_mask_value_profile_sample.csv", index=False)
display(mask_value_profile_df.head(80))
display(mask_value_profile_df.groupby(["gt_type", "modality", "value"], dropna=False)["ratio"].median().reset_index().sort_values(["gt_type", "modality", "ratio"], ascending=[True, True, False]).head(80))


In [ ]:

# 6) Construcción de candidatos: estrategia oficial por slice_number + fallback de revisión
def read_dicom(path):
    try:
        ds = pydicom.dcmread(str(path), force=True)
        _ = ds.pixel_array
        return ds
    except Exception:
        return None

img = image_df.copy()
gt = gt_png.copy()
img["instance_number_num"] = pd.to_numeric(img["instance_number"], errors="coerce").astype("Int64")
gt["disc_id_num"] = pd.to_numeric(gt["disc_or_slice_id"], errors="coerce").astype("Int64")

candidate_rows = []
if len(slices_long_df):
    sx = slices_long_df.copy()
    sx["case_id"] = sx["case_id"].apply(norm_case)
    sx["slice_number_num"] = pd.to_numeric(sx["slice_number"], errors="coerce").astype("Int64")
    sx["disc_id_num"] = pd.to_numeric(sx["disc_id"], errors="coerce").astype("Int64")
    sx["modality"] = sx["modality"].fillna("T1")

    m1 = img.merge(
        sx,
        left_on=["case_id_norm", "modality", "instance_number_num"],
        right_on=["case_id", "modality", "slice_number_num"],
        how="inner",
        suffixes=("_img", "_slice")
    )
    m2 = m1.merge(
        gt,
        left_on=["case_id_norm", "modality", "disc_id_num"],
        right_on=["case_id_norm", "modality", "disc_id_num"],
        how="inner",
        suffixes=("", "_gt")
    )
    for _, r in m2.iterrows():
        candidate_rows.append({
            "strategy": "official_slice_number",
            "case_id": r["case_id_norm"],
            "modality": r["modality"],
            "disc_id": int(r["disc_id_num"]) if pd.notna(r["disc_id_num"]) else None,
            "instance_number": int(r["instance_number_num"]) if pd.notna(r["instance_number_num"]) else None,
            "image_file_path": r["image_file_path"],
            "gt_file_path": r["gt_file_path"],
            "gt_type": r.get("gt_type"),
            "evidence": f"{r.get('source_table')} row={r.get('source_row')} col={r.get('source_column')} val={r.get('source_value')}",
        })

fallback = (
    img[img["case_id_norm"].notna() & img["modality"].notna()]
    .sort_values(["case_id_norm", "modality", "instance_number_num"])
    .groupby(["case_id_norm", "modality"], group_keys=False)
    .head(12)
    .merge(
        gt[gt["case_id_norm"].notna() & gt["modality"].notna() & gt["disc_id_num"].notna()]
        .sort_values(["case_id_norm", "modality", "gt_type", "disc_id_num"])
        .groupby(["case_id_norm", "modality", "disc_id_num"], group_keys=False)
        .head(2),
        on=["case_id_norm", "modality"],
        how="inner",
        suffixes=("_img", "_gt")
    )
)
for _, r in fallback.head(1500).iterrows():
    candidate_rows.append({
        "strategy": "fallback_case_modality_disc_review",
        "case_id": r["case_id_norm"],
        "modality": r["modality"],
        "disc_id": int(r["disc_id_num"]) if pd.notna(r["disc_id_num"]) else None,
        "instance_number": int(r["instance_number_num"]) if pd.notna(r["instance_number_num"]) else None,
        "image_file_path": r["image_file_path"],
        "gt_file_path": r["gt_file_path"],
        "gt_type": r.get("gt_type"),
        "evidence": "fallback; requiere revisión visual/manual",
    })

official_candidates_df = pd.DataFrame(candidate_rows).drop_duplicates(["image_file_path", "gt_file_path", "strategy"])
official_candidates_df.to_csv(E8_ROOT / "E8_official_pairing_candidates_raw.csv", index=False)
print("official_candidates_df:", official_candidates_df.shape)
display(official_candidates_df.head(50))
if len(official_candidates_df):
    display(official_candidates_df.groupby(["strategy", "modality", "gt_type"], dropna=False).size().reset_index(name="n").sort_values("n", ascending=False))


In [ ]:

# 7) Visualizar overlays por valor/clase, no como máscara binaria global
def normalize_image(arr):
    arr = arr.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr)
    return (np.clip(arr, p1, p99) - p1) / (p99 - p1 + 1e-8)

def mask_for_value(arr, value):
    if arr.ndim == 3:
        v = np.array(eval(value) if value.startswith("(") else value)
        return np.all(arr[..., :len(v)] == v, axis=-1).astype(np.uint8)
    return (arr == int(value)).astype(np.uint8)

def top_candidate_values(mask_path, min_ratio=0.0005, max_ratio=0.35, max_values=5):
    arr = read_mask_raw(mask_path)
    shape, prof = value_profile(mask_path, max_values=50)
    out = []
    # excluir valor dominante como background probable
    for item in prof[1:]:
        if min_ratio <= item["ratio"] <= max_ratio:
            out.append(item["value"])
        if len(out) >= max_values:
            break
    return arr, out

vis_source = official_candidates_df.copy()
vis_source["strategy_priority"] = vis_source["strategy"].map({"official_slice_number": 0, "fallback_case_modality_disc_review": 1}).fillna(9)
vis_source["modality_priority"] = vis_source["modality"].map({"T1": 0, "T2": 1}).fillna(9)
vis_source["gt_priority"] = vis_source["gt_type"].map({"final": 0, "intermediary": 1, "manual": 2}).fillna(9)
vis_df = (
    vis_source.sort_values(["strategy_priority", "modality_priority", "gt_priority", "case_id", "disc_id", "instance_number"])
    .groupby(["case_id", "modality", "disc_id"], group_keys=False)
    .head(2)
    .head(120)
    .copy()
)

review_rows = []
print("Candidatos a visualizar:", len(vis_df))
for i, (_, r) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc="official overlays"), start=1):
    cid = f"official_candidate_{i:03d}"
    try:
        ds = read_dicom(r["image_file_path"])
        if ds is None:
            raise RuntimeError("No se pudo leer DICOM")
        img_pixels = ds.pixel_array.astype(np.float32)
        img_norm = normalize_image(img_pixels)

        raw_mask, values = top_candidate_values(r["gt_file_path"])
        ncols = 3 + min(len(values), 4)
        fig, axes = plt.subplots(1, ncols, figsize=(4*ncols, 4))
        axes[0].imshow(img_norm, cmap="gray"); axes[0].set_title("Axial IMA"); axes[0].axis("off")
        axes[1].imshow(raw_mask if raw_mask.ndim == 3 else raw_mask, cmap=None if raw_mask.ndim == 3 else "nipy_spectral")
        axes[1].set_title("Raw GT"); axes[1].axis("off")

        combined = np.zeros(raw_mask.shape[:2], dtype=np.uint8)
        value_stats = []
        for v in values:
            m = mask_for_value(raw_mask, v)
            if m.shape != img_pixels.shape:
                m = resize(m.astype(np.float32), img_pixels.shape, order=0, preserve_range=True, anti_aliasing=False) > 0.5
                m = m.astype(np.uint8)
            ratio = float(m.mean())
            _, comps = ndimage.label(m > 0)
            combined = np.maximum(combined, m)
            value_stats.append({"value": v, "ratio": ratio, "components": int(comps)})

        axes[2].imshow(img_norm, cmap="gray")
        axes[2].imshow(np.ma.masked_where(combined <= 0, combined), alpha=0.45, cmap="autumn")
        axes[2].set_title("Overlay valores candidatos"); axes[2].axis("off")

        for ax, stat in zip(axes[3:], value_stats[:4]):
            m = mask_for_value(raw_mask, stat["value"])
            if m.shape != img_pixels.shape:
                m = resize(m.astype(np.float32), img_pixels.shape, order=0, preserve_range=True, anti_aliasing=False) > 0.5
            ax.imshow(img_norm, cmap="gray")
            ax.imshow(np.ma.masked_where(m <= 0, m), alpha=0.55, cmap="autumn")
            ax.set_title(f"value={stat['value']}\nratio={stat['ratio']:.3f}")
            ax.axis("off")

        fig.suptitle(
            f"{cid} | {r['strategy']} | case={r['case_id']} | {r['modality']} | D={r['disc_id']} | inst={r['instance_number']} | gt={r['gt_type']}",
            fontsize=10
        )
        fig.tight_layout()
        fig_path = FIGURES_ROOT / f"E8_alkafri_official_pairing_{i:03d}.png"
        fig.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        review_rows.append({
            "candidate_id": cid,
            "figure_path": str(fig_path),
            "strategy": r["strategy"],
            "case_id": r["case_id"],
            "modality": r["modality"],
            "disc_id": r["disc_id"],
            "instance_number": r["instance_number"],
            "gt_type": r["gt_type"],
            "image_file_path": r["image_file_path"],
            "gt_file_path": r["gt_file_path"],
            "candidate_values": json.dumps(value_stats, ensure_ascii=False),
            "auto_status": "review",
            "manual_accept": "",
            "manual_reject_reason": "",
            "notes": "",
        })
    except Exception as exc:
        review_rows.append({
            "candidate_id": cid,
            "strategy": r.get("strategy"),
            "case_id": r.get("case_id"),
            "modality": r.get("modality"),
            "disc_id": r.get("disc_id"),
            "instance_number": r.get("instance_number"),
            "gt_type": r.get("gt_type"),
            "image_file_path": r.get("image_file_path"),
            "gt_file_path": r.get("gt_file_path"),
            "auto_status": "error",
            "error": repr(exc),
            "manual_accept": "",
            "manual_reject_reason": "",
            "notes": "",
        })

review_df = pd.DataFrame(review_rows)
review_df.to_csv(E8_ROOT / "E8_official_pairing_visual_review_sheet.csv", index=False)
print("Figuras generadas:", int(review_df["figure_path"].notna().sum()) if "figure_path" in review_df else 0)
display(review_df.head(30))


In [ ]:

# 8) Reporte técnico y decisión
n_official = int((official_candidates_df["strategy"] == "official_slice_number").sum()) if len(official_candidates_df) else 0
n_fallback = int((official_candidates_df["strategy"] == "fallback_case_modality_disc_review").sum()) if len(official_candidates_df) else 0
n_figs = int(review_df["figure_path"].notna().sum()) if "figure_path" in review_df else 0

decision = "pending_manual_review"
recommendation = (
    "Abrir E8_official_pairing_visual_review_sheet.csv y marcar manual_accept=yes "
    "solo si el overlay por valor/clase cae anatómicamente sobre estructuras esperadas. "
    "Si hay >=30 aceptados, crear subset axial curado. Si no, cerrar Al-Kafri como no validado."
)

report = {
    "notebook": "16_E8_alkafri_official_pairing_and_mask_decode",
    "goal": "rescue axial segmentation pairing using official metadata and mask value profiling",
    "metadata_tables_found": list(metadata_tables.keys()),
    "slices_long_rows": int(len(slices_long_df)),
    "image_index_rows": int(len(image_df)),
    "gt_index_rows": int(len(gt_df)),
    "gt_png_rows": int(len(gt_png)),
    "official_slice_number_candidates": n_official,
    "fallback_review_candidates": n_fallback,
    "visual_review_figures": n_figs,
    "decision": decision,
    "recommendation": recommendation,
}

(E8_ROOT / "E8_alkafri_official_pairing_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

lines = [
    "# E8 Al-Kafri official pairing and mask decode rescue",
    "",
    "## Objetivo",
    "Intentar rescatar el dataset axial Al-Kafri/Sudirman usando metadata oficial y perfiles reales de valores de máscara.",
    "",
    "## Resultados",
    f"- Metadata CSV encontrada: {len(metadata_tables)}",
    f"- Filas detectadas en Slices Numbers long: {len(slices_long_df)}",
    f"- Imágenes indexadas: {len(image_df)}",
    f"- GT indexados: {len(gt_df)}",
    f"- GT PNG: {len(gt_png)}",
    f"- Candidatos por metadata oficial slice number: {n_official}",
    f"- Candidatos fallback para revisión: {n_fallback}",
    f"- Figuras para revisión visual: {n_figs}",
    "",
    "## Decisión",
    decision,
    "",
    "## Recomendación",
    recommendation,
]
(DOCS_ROOT / "E8_alkafri_official_pairing_conclusion.md").write_text("\n".join(lines), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
